## TRIVAL Stage 1

This code is provided as an example code to implement Thematic Review with inbuilt validation (TRIVAL) Stage 1. The users are free to download and make the changes based on their requirement. This exercise implements Bert Topic to identify the themes across various abstracts. However, with little modification users can implement Top2Vec and other Deep Learning based algorithms to implement the framework.

To execute the code. the user has to create profile at huggingface (https://huggingface.co/) and generate an access token (API key). The user will not be able to execute the code chunks without it.



## Installation

The use of the framework requires several packages to be installed. The package installation gives several issues (mostly package dependency related issues). Generally, the below code chunk will install all the packages needed.

The installations are one time activity. We do not need to run the instllation block everytime we use this notebook. The code blocks in this section are commented out. Remove "#" to uncomment the block. The lines with double hash "##" are just code comments. Do not remove the hash symbol from there.

In [ ]:
#!pip install transformers

In [ ]:
#!pip install -U bertopic datasets accelerate bitsandbytes xformers adjustText

### Additional step (Optional)

These steps are not needed, if the above code block runs and installs all the packages (check the log above to see if there are any errors in installation).

In case of error, copy the error and ask for help through Gemini available on the upper right hand corner. Gemini will give the correct code to run. Like, in our case, we needed to perform the below steps for successful installation of all the packages. But it could be different for different users. Generally, the below code will work. After uninstalling a package, restart the notebook before installing new packages.

In [ ]:
## Uninstall conflicting packages
#!pip uninstall -y fsspec torch torchaudio torchvision fastai timm gcsfs

In [ ]:
#!pip install fsspec==2025.3.2
#!pip install gcsfs==2023.3.0

In [ ]:
# If timm 1.0.15 requires torchvision 0.16.1 (this is an example, check the actual requirement)
#!pip install torchvision>=0.16.1
#!pip install timm==1.0.15

In [ ]:
#!pip install torch
#!pip install torchaudio
#!pip install fastai

In [ ]:
#!pip install transformers

In [ ]:

#!pip install -U bertopic datasets accelerate bitsandbytes xformers adjustText

## Create Credentials

We will use Llama model for which APIs are provided by Hugging Face. So first we need an account in Hugging Face.


* Create a HuggingFace account https://huggingface.co/

Before we can load Llama2, we will first need to accept the License for using Llama2.

* Apply for Llama 2 access https://huggingface.co/meta-llama/Llama-2-13b-chat-hf

Note that, we need to do the above step anytime, we need to use a new model. It requires filling a form and submitting. The access is granted within 1-2 days.

Next, create a token to call the modeles hosted in Hugging Face. This is a one time activity. Please do not share the token with anyone else.

* Get your HuggingFace token https://huggingface.co/settings/tokens

After doing so, we can log in with the HuggingFace credentials so that the environment knows we have permission to download the Llama 2 model that we are interested to use here.

We have all the pre-requisites to run the code now. The below code is used to login to hugging face using the user genereated token.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

## HF_API_KEY need to be replaced with user genereated API key at hugging face
os.environ["HF_TOKEN"] = userdata.get("HF_API_KEY")
login(token=os.environ["HF_TOKEN"])

## Import Data

There are several ways to import the file. However, we will save the file in google drive and read the file in this notebook. Once the files are uploaded, the below code chunk helps mount the google drive for colab use. It will ask for permission to access google drive. Users must grant permission by giving their signin credentials.

In [ ]:
from google.colab import drive
drive.mount('/content/files')
#drive.flush_and_unmount()

We create a folder in googel drive named "BertWithLlama2" inside which the TRIVALStage1.ipnb and TRIVALStage2.ipynb files is available along with the file that contains abstract and other metadata associated with it. We have named this file as "rawdata694_08082023.csv". Users are free to rename it. Once the access is provided, the files can be accessed by giving the pathname as "/content/files/MyDrive/BertWithLlama2/"

In [ ]:
path_name = "/content/files/MyDrive/BertWithLlama2/"
#to change the dir to path_name
os.chdir(path_name)
#to re-print the dir name
os.getcwd()

We are using pandas library to read the files. So we import pandas package using the below code line and then use read_csv() to import the rawdata694_08082023.csv. The sccessful read of the file will show the first five rows of information from the file.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('./rawdata694_08082023.csv', na_values =['[No abstract available]', ''])
df.head()

A quick look at the column names suggests that we are interested only in title and abstract column. We will pull these two columns into other variable for doing thematic analysis.

In [ ]:
df.info()

In [ ]:
# Extract abstracts to train on and corresponding titles
titles = df["Title"].fillna("")  # Fill NaN values with empty string
abstracts = df["Abstract"].fillna("") # Fill NaN values with empty string

A sample abstract look likes

In [ ]:
print(abstracts[120])

## Import Llama2

Now we load the LLM - Llama2 on a T4-GPU which is provided for free by Google Colab.

We will be focusing on the 'meta-llama/Llama-2-13b-chat-hf' variant. It is large enough to give interesting and useful results whilst small enough that it can be run on our environment.

### Optimization & Quantization

To use a very large AI model, which is like a massive brain with 13 billion connections, we need to make it smaller so it fits in our computer's memory. Think of it like taking a huge book and summarizing it so you can quickly grasp the main ideas without reading every single word.

We do this by using a special technique called "4-bit quantization."

Imagine each piece of information in the AI model is like a detailed instruction written with 64 numbers. 4-bit quantization is like rewriting those instructions using only 4 numbers. This significantly shrinks the size of the model, making it easier to load and use, especially when our computer has limited memory. It's a clever way to keep the model powerful while making it more practical to run. The two code chunks below exactly helps in doing that. The four parameters that are used in below code are important and bring many LLM applications to consumers:

* **load_in_4bit** - Allows us to load the model in 4-bit precision compared to the original 32-bit precision. This gives us an incredible speed up and reduces memory!

* **bnb_4bit_quant_type** - This is the type of 4-bit precision. The paper recommends normalized float 4-bit, so that is what we are going to use!

* **bnb_4bit_use_double_quant** - This is a neat trick as it performs a second quantization after the first which further reduces the necessary bits

* **bnb_4bit_compute_dtype** - The compute type used during computation, which further speeds up the model.

Using this configuration, we can start loading in the model as well as the tokenizer

We start by defining our model in model_id and identifying if our GPU is correctly selected. We expect the output of device to show a Cuda device: 0. If the output shows "cpu", then change the runtime to t4 GPU from the colab settings on the upper right side icon. It may also be likely, that the runtime cannot be set to GPU if the free limit is exceeded. Either wait for 1-2 days or buy the GPU units from Google to proceed.

In [ ]:
from torch import cuda

#Usage of Llama requires GPU. Swtich to GPU mode (Edit -> Notebook Settings)
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'; print(device)

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from torch import bfloat16
import transformers

# Quantization to load an LLM with less GPU memory
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

Now we are ready to load the model as well as the tokenizer needed to use LLama2. Every model like LLama2, comes with its own tokenizer. In simple terms tokenizers convert a word into number which is dependent on the model being used. model_id variable contains the model being used. The dependent tokenizer is loaded by below code chunk.

In [ ]:
## Usage of Llama requires GPU. Swtich to GPU mode (Edit -> Notebook Settings)
model_id = 'meta-llama/Llama-2-13b-chat-hf'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

We also load the model Llama2 13 billion parameter model using below code.

In [ ]:
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

Now the model and tokenizer is loaded for use. We will create a transformers pipeline that allows us to easily generate new text. Pipeline can be thought of as sequence of steps stored in a variable (called generator here) which is used by the LLMs to generate texts.

In [ ]:
# Our text generator
generator = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task='text-generation',
    temperature=0.1,
    max_new_tokens=500,
    repetition_penalty=1.1
)

To check whether our model is correctly loaded, let’s try it out with a few prompts.

In [ ]:
prompt = "Please explain 4-bit quantization to a management scholar?"
res = generator(prompt)
print(res[0]["generated_text"])

Great! The model is loaded as we were able to generate the above text. This is similar to how we write text in ChatGPT and get an output. The difference here is that we are doing it through code and we can use any LLM we want (including the OpenAI model - ChatGPT).

But to get more meangiful outcome. We need to do some prompt engineering.

## Prompt Engineering

Although we can directly prompt the model, there is actually a template that we need to follow. The template looks as follows:

In [ ]:
"""
<s>[INST] <<SYS>>

{{ System Prompt }}
<</SYS>>
{{ User Prompt }}
 [/INST]
{{ Model Answer }}
"""

### Prompt Template

With the main goal of topic labeling, we will use a simple system prompt that defines the model as a helpful assistant for this task.

This template consists of two main components, namely the {{ System Prompt }} and the {{ User Prompt }}:

1. The {{ System Prompt }} helps us guide the model during a conversation. For example, we can say that it is a helpful assistant that is specialized in labeling topics.
2. The {{ User Prompt }} is where we ask it a question.

The [INST] tags are used to identify the beginning and end of a prompt. We can use these to model the conversation history as well which we will see later on. Below code line defines a system prompt.

In [ ]:
# System prompt describes information given to all conversations
system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful, honest assistant for labeling topics.
<</SYS>>
"""

In contrast, the user prompt is going to be a bit more involved. It will consist of two components, an example prompt and the main prompt.

Most LLMs do a much better job of generating accurate responses if you give them an example to work with. We provide the LLM with five abstracts below and thus it can be called as a "few shot learner". We also show it an accurate example of the kind of output we are expecting. Like giving it the list of keywords from the five examples as well as the name of the theme it should derive from the keywords. Here we gave the theme name as "Building an yoga expert using deep learning algorithm". Thus we showed the model that we were expecting only the label.

In [ ]:
# Example prompt demonstrating the output we are looking for
example_prompt = """
I have a topic that contains the following documents:
- Yoga is a traditional Indian exercise. It specifies various body postures called asanas, practicing them is beneficial for the physical, mental, and spiritual well-being. To support the yoga practitioners, there is a need of an expert yoga asanas recognition system that can automatically analyze practitioner's postures and could provide suitable posture correction instructions. This paper proposes YogNet, a multi-person yoga expert system for 20 asanas using a two-stream deep spatiotemporal neural network architecture. The first stream utilizes a keypoint detection approach to detect the practitioner's pose, followed by the formation of bounding boxes across the subject. The model then applies time distributed convolutional neural networks (CNNs) to extract frame-wise postural features, followed by regularized long short-term memory (LSTM) networks to give temporal predictions. The second stream utilizes 3D-CNNs for spatiotemporal feature extraction from RGB videos. Finally, the scores of two streams are fused using multiple fusion techniques. A yoga asana recognition database (YAR) containing 1206 videos is collected using a single 2D web camera for 367 min with the help of 16 participants and contains four view variations i.e. front, back, left, and right sides. The proposed system is novel as this is the earliest two-stream deep learning-based system that can perform multi-person yoga asanas recognition and correction in realtime. Simulation result reveals that YogNet system achieved 77.29%, 89.29%, and 96.31% accuracies using pose stream, RGB stream, and via fusion of both streams, respectively. These results are impressive and sufficiently high for recommendation towards general adaption of the system.
- People are deepening the study on model explainability as well as performance to better understand models’ decisions from the human perspective. However, the lack of rare clinical diagnosis data always limits the power of the emerging data-driven deep diagnosis methods, and the traditional deep transfer learning (DTL) applicable for the case is often insufficient to learn specific features in medical image processing, leading to poor explainability. To address those challenges, a two-stage deep transfer learning model is proposed and applied to assist in the Traditional Chinese Medicine (TCM) tongue diagnosis. Especially, a two-stage transfer learning training strategy is designed to loose the data dependence of deep learning on the domain data, which is composed of the imitate stage that discovers shared basic source features and the transfer stage to relearn target patterns, with good explainability. The corresponding deep squeeze-and-excitation convolutional network is proposed to learn the clinical patterns of tongue symptoms, in which a three-layer feature pyramid network fuses the multi-scale tongue features. Extensive experiments are conducted on the real clinical dataset in terms of classification accuracy and learning efficiency. The resulting accuracy of the proposed model proves its performance advantage with the recognition time achieving real-time performance.
- Management learning scholarship has acknowledged how learning relates to bodies and embodied practices. Yet the mechanisms through which abstract management concepts are built from embodied practice require further exploration. How individuals work reflexively through bodily practices in their evolving understandings of management, and the implications of these practices for learner agency and control of the body, are not well-understood. By examining an empirical case of a yoga-based leadership course, we explore how reflexive embodied practices ground leadership concept development. Focusing on embodied analogy as our analytical lens, we explore how participants mapped abstract concepts through bodily postures, inner sensations of bodily intensity, and reflection on bodily experience. We explore three mechanisms by which embodied leadership concepts were constructed-performative analogization, intentionality through bodily intensity, and unlearning by analogization-highlighting the role of analogy in enacting reflexive embodied practices around leadership. We discuss the implications for pedagogies of embodiment, while theorizing the critical implications of techniques of the body for understanding management learning.
- Flow as propounded by Csikszentmihalyi is an extraordinary phenomenon helping people maintaining all-encompassing performance ability often triggers curiosity of whether that particular phenomenon can be replicated time and again amongst all the humans. Well, the answer can be both a YES and a NO. In order to explore the phenomenon, we will explore the principles or laws of the universe and its functioning as propounded by Gurdjieff and draw assertions from the exploration to design applicable understanding in the nature of flow. The understanding of Gurdjieff was based on his search for the truth and the mysteries present within the mystic schools in the East. Much of the spiritual work is based on this basic premise that life propagates itself and the understanding of this phenomenon is self-evident to be propounded in the basic dogmatic revelations of the individual agenda. The fundamental truth about the expectation and reality dilemma about the performance scenario based on the interaction point of view with human and its capacities along with the environmental stimuli often brings around certain understanding about the capacities and performance ability in a certain environmental setup. This particular transcending phenomena and immersion and evolving in the process of working and making extraordinary revolutions in all the spheres of life has been a constant curious case to be solved and debated with the help of yoga, Zen, science, religion, mysticism, and occult culminating from various forms of old age and new age philosophical dimensions engulfed into the spiritual underpinnings. Transcending above one's own physical boundaries to take into to the cosmic vibrations and inculcate the energy field within one's own boundaries helping them to perform and replicate things in finest creativity levels often motivates to explore these phenomena into further details.
- Alternative medicine is one of the ancient traditional treatments in India which has been accepted worldwide. According to World Health Organization (WHO) the use of Alternative medicine is in rise. India has a population of around 132.42 Crore as of 2018 and there is a lack of awareness of the Alternate medicine in this demographic system. In NHM to seek affordability; provide accessible and standard health care to improve the healthcare system, Alternative medicine is one of the strategies. In this study we explore the database of AYUSH (Ayurveda, Unani, Siddha, naturopathy, Yoga, homeopathy). Government is working hard to bring the awareness among the people by conducting some health campaigns. So, the present conceptual study will be undergoing in the aspect of finding the variables for the Perception of customers (doctors and patients) regarding Alternative medicine.

The topic is described by the following keywords: 'computer vision, posture correction, yoga and exercise, deep neural networks, deep transfer learning, medical image processing, tongue diagnosis, traditional chinese medicine, leadership concept, yoga and embodied practices, performative analogization, unlearning by analogization, religion, Creativity level, Law of Gurdjieff, Skill-Work match, alternative medicine, AYUSH-Ayurveda, health tourism, homeopathy, siddha, Unani, Yoga'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
[/INST] Building an yoga expert using deep learning algorithm.
"""

In [ ]:
# Example prompt demonstrating the output we are looking for
#example_prompt = """
#I have a topic that contains the following documents:
#- Yoga is a traditional Indian exercise. It specifies various body postures called asanas, practicing them is beneficial for the physical, mental, and spiritual well-being.
#- To support the yoga practitioners, there is a need of an expert yoga asanas recognition system that can automatically analyze practitioner's postures and could provide suitable posture correction instructions.
#- This paper proposes YogNet, a multi-person yoga expert system for 20 asanas using a two-stream deep spatiotemporal neural network architecture. The first stream utilizes a keypoint detection approach to detect the practitioner's pose, followed by the formation of bounding boxes across the subject.
#- The model then applies time distributed convolutional neural networks (CNNs) to extract frame-wise postural features, followed by regularized long short-term memory (LSTM) networks to give temporal predictions.
#- The second stream utilizes 3D-CNNs for spatiotemporal feature extraction from RGB videos. Finally, the scores of two streams are fused using multiple fusion techniques.
#- A yoga asana recognition database (YAR) containing 1206 videos is collected using a single 2D web camera for 367 min with the help of 16 participants and contains four view variations i.e. front, back, left, and right sides. The proposed system is novel as this is the earliest two-stream deep learning-based system that can perform multi-person yoga asanas recognition and correction in realtime. Simulation result reveals that YogNet system achieved 77.29%, 89.29%, and 96.31% accuracies using pose stream, RGB stream, and via fusion of both streams, respectively. These results are impressive and sufficiently high for recommendation towards general adaption of the system.


#The topic is described by the following keywords: 'computer vision, posture correction, yoga and exercise, deep neural networks, LSTM, CNN, well being'.

#Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
#[/INST] Building an yoga expert for physical well being using deep learning algorithm.
#"""

This example, based on a number of keywords and documents primarily about the impact of meat, helps the model understand the kind of output it should give.There are two BERTopic-specific tags that are of interest, namely [DOCUMENTS] and [KEYWORDS]:

1. [DOCUMENTS] contain the top 5 most relevant documents to the topic

2. [KEYWORDS] contain the top 10 most relevant keywords to the topic as generated through c-TF-IDF

In [ ]:
# Our main prompt with documents ([DOCUMENTS]) and keywords ([KEYWORDS]) tags
main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
[/INST]
"""



This template will be filled according to each topic. And finally, we can combine this into our final prompt:

In [ ]:
prompt = system_prompt + example_prompt + main_prompt

## Topic modelling using BERTopic
Before we can start with topic modeling, we will first need to perform two steps:

* Pre-calculating Embeddings

* Defining Sub-models

### Preparing Embeddings
By pre-calculating the embeddings for each document, we can speed up additional exploration steps and use the embeddings to quickly iterate over BERTopic’s hyperparameters if needed.

🔥 TIP: You can find a great overview of good embeddings for clustering on the MTEB Leaderboard.

In [ ]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

## Understanding Umap and Hdbscan

Imagine you're trying to group customer feedback into meaningful categories or themes. If each piece of feedback has many characteristics (like length, sentiment, mentions of specific products, etc.), it's like having data in a high-dimensional space.

HDBSCAN is a method that tries to find clusters of feedback based on how dense the feedback is in this multi-dimensional space. However, when you have a lot of dimensions (the "curse of dimensionality"), the data points can become so spread out that it's hard to see where the dense areas are. You need a massive amount of data to overcome this.

Think of it like trying to spot patterns in customer behavior when you're tracking dozens of different metrics for each customer. The sheer number of metrics makes it difficult to see clear groups.

One way to deal with this is to simplify the data by reducing the number of dimensions. Principal Component Analysis (PCA) is a common technique for this. It's like summarizing the most important characteristics of each piece of feedback. However, PCA works best when the relationships between these characteristics are straightforward (linear). In reality, customer feedback might have more complex, non-linear relationships between characteristics. If you reduce the dimensions too much with PCA, you might lose important information.

This is where UMAP comes in. UMAP is better at capturing these complex, non-linear relationships when reducing dimensions. It's like finding the underlying themes in customer feedback that aren't immediately obvious from individual characteristics. UMAP helps to bring the data closer together in areas where it's truly similar, making the dense clusters more apparent for HDBSCAN.

Unlike methods used just for visualizing data (which often reduce to just two dimensions), UMAP allows you to reduce to a higher number of dimensions (like 10) while still making the data easier for HDBSCAN to cluster. This is beneficial because our goal is to find meaningful groups, not just to create a pretty picture.

When using UMAP to prepare data for clustering with HDBSCAN, you'll adjust its settings differently than when you're just trying to visualize the data. You might increase the n_neighbors setting. This helps UMAP focus on broader patterns of density, rather than getting stuck on very small, potentially noisy clusters that might not be truly meaningful in your customer data. You might also set min_dist to a very low value. This encourages UMAP to pack truly similar data points together more tightly, making the boundaries between clusters clearer. You can read more about HDBSCAN and UMAP at:

* https://umap-learn.readthedocs.io/en/latest/clustering.html
* https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html
* https://pberba.github.io/stats/2020/01/17/hdbscan

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
import seaborn as sns
sns.set_context('poster')
sns.set_style('white')
sns.set_color_codes()
plot_kwds = {'alpha' : 0.5, 's' : 80, 'linewidths':0}


import hdbscan
import umap

import logging
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#logging.captureWarnings(True)
clusterable_embedding = umap.UMAP(
    n_neighbors=40,
    min_dist=0.0,
    n_components=2,
    random_state=42,
).fit_transform(embeddings)

We can visualize the results of this so see how it compares with more visualization attuned parameters:


In [ ]:
plt.scatter(clusterable_embedding[:, 0], clusterable_embedding[:, 1],
            color=(0.5, 0.5, 0.5), s=.8, cmap='Spectral', alpha = 1.0);


After applying a data reduction technique (like summarizing complex data into fewer key aspects), we can see the overall structure of our data is maintained. However, for the purpose of grouping similar data points together (clustering), we've compressed the data within these groups more tightly. This creates clearer boundaries, or "gaps," between the potential clusters.

This reduced representation of the data was specifically prepared to help a clustering method called HDBSCAN identify groups. For visualizing the data, we'll revert to the original, less compressed representation. The next step involves using HDBSCAN to find these clusters. We'll use the same settings as applied previously.

In HDBSCAN, a key parameter is min_samples. Imagine we are trying to understand how concentrated our data is in different areas. min_samples helps us define what we consider a "dense" area. It essentially looks at how far we need to go to find a certain number of neighboring data points.

* If min_samples is too low, we become very sensitive to minor variations in the data density. This can lead to finding many small, potentially meaningless groups, or even splitting up larger, meaningful groups.
* If min_samples is too high, we might smooth over important details in the data and only identify the most prominent, large-scale groups.

Think of min_samples as a control for how cautious you want to be in identifying clusters. A higher value means you're more conservative; more data points will be considered outliers (noise), and identified clusters will be limited to the most densely populated areas.

Another important parameter is min_cluster_size. This parameter sets a threshold for the minimum number of data points required for a collection of points to be considered a valid cluster.

* By increasing min_cluster_size, you're essentially smoothing out the "landscape" of the data's density, making only the most significant "peaks" (dense regions) stand out as potential clusters.

While min_samples helps estimate the overall density, min_cluster_size helps ensure that only substantial dense regions are recognized as clusters, especially when we have a good estimate of the data's density from the min_samples setting.

In [ ]:
hdb = hdbscan.HDBSCAN(
                  #min_sample is k nearest neighbour
                  min_samples=5,
                  #Max size of the bump before it can be considered as a peak.
                  min_cluster_size=5,gen_min_span_tree=True).fit(clusterable_embedding)

And now we can visualize the results, just as before.

In [ ]:
#if we uncomment the previous line, then we have to use labels rather than hdb.labels_
clustered = (hdb.labels_ >= 0)
plt.scatter(clusterable_embedding[~clustered, 0],
            clusterable_embedding[~clustered, 1],
            color=(0.5, 0.5, 0.5),
            s=0.8,
            alpha=1.0)
plt.scatter(clusterable_embedding[clustered, 0],
            clusterable_embedding[clustered, 1],
            c=hdb.labels_[clustered],
            s=0.8,
            cmap='Spectral');

How much more of the data is HDBSCAN actually clustering?

In [ ]:
clustered = (hdb.labels_>0)
coverage = np.sum(clustered) / clusterable_embedding.shape[0]

print(f"Coverage {coverage}")
print(f"DBCV score {hdb.relative_validity_}")

Now we are clustering nearly 65% of the data! But we see a very low DBCV  score. This means that we need to fine tune the parameters so as to get a better DBCV score.

## Finetuning Hyperparameters based on DBCV score


Density Based Clustering Validation or DBCV works for desnity-based clustering algorithms precisely because it takes noise into account and captures the shape property of clusters via densities and not distances. The final result of DBCV is a weighted sum of “Validity Index” values of clusters. This produces a score between -1 to 1, with the larger the value the better clustering solution. Reference paper:


* https://www.researchgate.net/publication/260333211_Density-Based_Clustering_Validation

* https://towardsdatascience.com/tuning-with-hdbscan-149865ac2970



In [ ]:
n_neighbors = 40
n_components = 5
umap_model = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=0.0,
    n_components=n_components,
    random_state=42,
).fit_transform(embeddings)

In [ ]:
# Naive grid search implementation by Mueller and Guido, Introduction to Machine Learning with Python

best_score = 0

for min_samples in [3,4,5,6,7,10,15,20,25,30,35,40,45,50,55]:
    for min_cluster_size in [3,4,5,6,7,10,15,20,25]:
        for cluster_selection_method in ['eom','leaf']:
            for metric in ['euclidean']:
                # for each combination of parameters of hdbscan
                hdb = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,min_samples=min_samples,
                                      cluster_selection_method=cluster_selection_method, metric=metric,
                                      gen_min_span_tree=True).fit(umap_model) #clusterable_embedding - o/p of UMPA, embeddings
                # DBCV score
                score = hdb.relative_validity_
                # if we got a better DBCV, store it and the parameters
                if score > best_score:
                    best_score = score
                    best_parameters = {'min_samples':  min_samples,
                                       'min_cluster_size': min_cluster_size,
                                       'cluster_selection_method': cluster_selection_method,
                                       'metric': metric}

print("Best DBCV score: {:.3f}".format(best_score))
print("Best parameters: {}".format(best_parameters))

The fine tunining is an iterative process in which we started by setting n_neighbors = 40 and n_component = 5 (this also was iterated). Post that we run the above grid to find the best set of parameters which improves DBCV score. The final result shown above

We use the above parameter setting to proceed with rest of the code for clustering the abstracts.

In [ ]:
n_neighbors=40
n_components=5
min_sample = 3
min_cluster_size = 3
cluster_selection_method: 'eom'
metric: 'euclidean'

In [ ]:
logging.captureWarnings(True)
clusterable_embedding = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=0.0,
    n_components=n_components,
    metric='cosine',
    random_state=42,
).fit_transform(embeddings)

hdb = hdbscan.HDBSCAN(
                  min_samples=min_sample,
                  min_cluster_size=min_cluster_size,gen_min_span_tree=True).fit(clusterable_embedding)

coverage = np.sum(hdb.labels_) / embeddings.shape[0]

print(f"Coverage {coverage}")
print(f"DBCV score {hdb.relative_validity_}")

As a small bonus, we are going to reduce the embeddings we created before to 2-dimensions so that we can use them for visualization purposes when we have created our topics.


In [ ]:
# Pre-reduce embeddings for visualization purposes
reduced_embeddings = umap.UMAP(n_neighbors=n_neighbors, n_components=2, min_dist=0.0,
                          metric='cosine', random_state=42).fit_transform(embeddings)

## Training - All the steps together

Now that we know the hyper paramters to be used for UMAP and HDBSCAN, we can start training the topic model! We supply BERTopic with the sub-models of interest, run .fit_transform, and get topics.

Useful links to understand Bert topic and dimensionality reduction such as umap

* https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html
* https://umap-learn.readthedocs.io/en/latest/clustering.html
* https://maartengr.github.io/BERTopic/getting_started/dim_reduction/dim_reduction.html


**Note**: The final set of themes may not exactly match with the one provided as part of the supplement material. One of the possible reason could be the update to the packages and dependent packages which keeps on optimizing the features. Also, some of the default parameters gets deprecated or the default values changes over a period of time. Some of these are beyond the control; unless users make a note of the exact version of the package used in the initial run of the code.


We load the needed packages.These are all the steps for bert topic.

In [ ]:
#from umap import UMAP
#from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer

### Representation Models for labeling the output

One of the ways we are going to represent the topics is with Llama 2 which should give us a nice label. However, we might want to have additional representations to view a topic from multiple angles.

Here, we will be using c-TF-IDF as our main representation and KeyBERT, MMR, and Llama 2 as our additional representations.

In [ ]:
# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

# Text generation with Llama 2
#llama2 = TextGeneration(generator, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    #"Llama2": llama2,
    "MMR": mmr
}

Bringing the steps together for topic modeling

In [ ]:
## Step 1 - Extract embeddings
## https://www.sbert.net/docs/pretrained_models.html
#embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

## Step 2 - Reduce dimensionality
## Based on fine-tuning earlier
## It was named as clusterable_embedding which we changed to umap_model

umap_model = umap.UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine')

## Step 3 - Cluster reduced embeddings
## Based on fine-tuning earlier
## min_sample = 3, min_cluster_size = 3
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples = min_sample, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = representation_model

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model, # Step 6 - (Optional) Fine-tune topic represenations

  # Hyperparameters
  top_n_words=10,
  #nr_topics = 'auto',
  #n_gram_range = c(1,1),
  verbose=True
)

The model training using above six steps.

In [ ]:
# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings)

Now that we are done training our model, let’s see what topics were generated:

In [ ]:
df1 = pd.DataFrame({ 'title': titles, 'abstract': abstracts,'clusterNo': topic_model.topics_})
df1.head()

We can save the above using the code below. Uncomment and run the code.

In [ ]:
#df1.to_csv("LLM_Param40_5_3_3.csv")
#files.download('LLM_Param40_5_3_3.csv')

We can see the top and bottom used topics in the corpus as below.

In [ ]:
# Show top 3 most frequent topics
topic_model.get_topic_info()[1:4]

In [ ]:
# Show top 3 least frequent topics
topic_model.get_topic_info()[-3:]

To save all the topics, we can uncomment the code below.

In [ ]:
#tp = topic_model.get_topic_info()
#tp.to_csv("topics_Param40_5_3_3.csv")
#files.download('topics_Param40_5_3_3.csv')

We got several topics that were created and they all seem quite diverse. We can use the labels by Llama 2 and assign them to topics that we have created. Normally, the default topic representation would be c-TF-IDF, but we will focus on Llama 2 representations instead.

In [ ]:
llama2_labels = [label[0][0].split("\n")[0] for label in topic_model.get_topics(full=True)["Llama2"].values()]
topic_model.set_topic_labels(llama2_labels)

## Visualize
We can go through each topic manually, which would take a lot of work, or we can visualize them all in a single interactive graph.

BERTopic has a bunch of visualization functions that we can use. For now, we are sticking with visualizing the documents.

In [ ]:
topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings,
hide_annotations=True, hide_document_hover=False, custom_labels=True)

